# MC occupancy for candidate reco binnings

The finer-binning study (`xml/binning_study/`) fits the NuWro fake data with more reco
log10(Q^2) x p_n bins than the production 9 x 8 grid. PROfit's `mcstat` covariance is built
from the inverse of the per-bin MC error, so a reco bin with no MC events is singular and a
nearly empty one is dominated by MC noise. Every study variant therefore has to keep at least
`MIN_MC_EVENTS` raw MC events in every bin.

This notebook shows how many MC events survive for the study variants and for any candidate
edges typed into Section 3, so a binning can be tuned by hand before it is added to the study.
It is self-contained: the variants are read from the reco `bins2D` edges of the XMLs below
`xml/binning_study/<variant>/`, so the table always reflects what the runner will fit.

| quantity | meaning |
|---|---|
| **MC raw** | unweighted selected overlay events (signal + background), the number that drives the `mcstat` error |
| **signal / background** | the same, split by `afro_1mu1p_true` |
| **expected** | GENIE CV prediction at the data POT (`net_weight`, scaled 9.3221e19 -> 1.75e21) |
| **NuWro raw** | unweighted selected NuWro fake-data events |

In [ ]:
from pathlib import Path
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

INPUT_FILE = '/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root'
MC_POT = 9.3221e19       # overlay POT in the MCFile block of the XMLs
DATA_POT = 1.75e21       # detector POT of the NuWro fake-data fits
POT_SCALE = DATA_POT / MC_POT

MIN_MC_EVENTS = 10                 # required raw MC events per reco bin for every study variant ...
THRESHOLD_EXEMPT = {'nominal'}     # ... except the production reference

# Production reco binning (xml/nuwro) and the p_n axis the finer variants start from: the
# production corner bin log10(Q^2) in [-2, -1.5] x p_n in [0, 0.1] holds only 5 MC events, so
# the finer variants move the first p_n edge from 0.10 to 0.15.
NOMINAL_Q2 = [-2.00, -1.50, -1.20, -1.00, -0.85, -0.70, -0.55, -0.40, -0.20, 0.20]
NOMINAL_PN = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 1.00]
STUDY_PN = [0.00, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 1.00]


def halve(edges, keep=()):
    '''Split every bin in two, except the bins whose index is listed in `keep` (negative = from the end).'''
    nbins = len(edges) - 1
    keep = {index % nbins for index in keep}
    out = []
    for index, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        out.append(lo)
        if index not in keep:
            out.append(0.5 * (lo + hi))
    out.append(edges[-1])
    return out


def fmt_edges(edges):
    '''Edges formatted as in the XML edgesx/edgesy attributes.'''
    def fmt(value):
        text = f'{value:.3f}'
        return text[:-1] if text.endswith('0') else text
    return ' '.join(fmt(edge) for edge in edges)


# Locate ma_zexp/xml/binning_study whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
STUDY_DIR = next(
    (parent / 'ma_zexp' / 'xml' / 'binning_study' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'xml' / 'binning_study').is_dir()),
    None,
)
if STUDY_DIR is None:
    raise FileNotFoundError('Could not locate ma_zexp/xml/binning_study')

RECO_UNIT = 'log10(Q^2 / GeV^2);p_n'


def read_reco_edges(xml_path):
    '''(log10 Q^2 edges, p_n edges) of the reco bins2D block of a PROfit XML.'''
    text = Path(xml_path).read_text()
    for block in re.findall(r'<bins2D\b.*?/>', text, flags=re.DOTALL):
        unit = re.search(r'unit="([^"]*)"', block)
        if unit and unit.group(1) == RECO_UNIT:
            edgesx = [float(v) for v in re.search(r'edgesx="([^"]*)"', block).group(1).split()]
            edgesy = [float(v) for v in re.search(r'edgesy="([^"]*)"', block).group(1).split()]
            return edgesx, edgesy
    raise ValueError(f'{xml_path}: no reco bins2D block with unit {RECO_UNIT!r}')


# variant name -> (log10 Q^2 edges, p_n edges), read from the first XML of each variant directory.
VARIANTS = {}
for variant_dir in sorted(STUDY_DIR.iterdir()):
    xmls = sorted(variant_dir.glob('*.xml'))
    if xmls:
        VARIANTS[variant_dir.name] = read_reco_edges(xmls[0])
if not VARIANTS:
    raise RuntimeError(f'no variant XMLs below {STUDY_DIR}; run python/scripts/make_binning_study_xmls.py')

print(f'MIN_MC_EVENTS = {MIN_MC_EVENTS}; exempt variants: {sorted(THRESHOLD_EXEMPT)}')
print(f'input: {INPUT_FILE}')
print(f'study XMLs: {STUDY_DIR}')
for name, (q2, pn) in VARIANTS.items():
    print(f'  {name:14s} {len(q2) - 1:2d} x {len(pn) - 1:2d} = {(len(q2) - 1) * (len(pn) - 1):3d} bins')

In [ ]:
# Load the reco variables once. The selection matches weight_1 of the XML branches.
tree = uproot.open(INPUT_FILE)['tree']
arrays = tree.arrays(
    ['isdata', 'isext', 'isdirt', 'isnuwro', 'afro_1mu1p_sel', 'afro_1mu1p_true',
     'afro_1mu1p_Q2', 'afro_1mu1p_Pn', 'net_weight'],
    library='np',
)
overlay = ((arrays['isdata'] == 0) & (arrays['isext'] == 0) & (arrays['isdirt'] == 0)
           & (arrays['isnuwro'] == 0) & (arrays['afro_1mu1p_sel'] == 1))
nuwro = (arrays['isnuwro'] == 1) & (arrays['afro_1mu1p_sel'] == 1)
with np.errstate(divide='ignore', invalid='ignore'):
    LOGQ2 = np.log10(arrays['afro_1mu1p_Q2'])
PN = arrays['afro_1mu1p_Pn']

SAMPLES = {
    'mc': (overlay, None),
    'signal': (overlay & (arrays['afro_1mu1p_true'] == 1), None),
    'background': (overlay & (arrays['afro_1mu1p_true'] == 0), None),
    'expected': (overlay, arrays['net_weight'] * POT_SCALE),
    'nuwro': (nuwro, None),
}
print(f"selected overlay events: {overlay.sum()}  (signal {SAMPLES['signal'][0].sum()}, "
      f"background {SAMPLES['background'][0].sum()});  NuWro: {nuwro.sum()}")

In [ ]:
def occupancy(q2_edges, pn_edges):
    '''Per-bin 2D histograms (rows = log10 Q^2 bins, columns = p_n bins) for every sample.'''
    out = {}
    for name, (mask, weights) in SAMPLES.items():
        out[name] = np.histogram2d(
            LOGQ2[mask], PN[mask], bins=[q2_edges, pn_edges],
            weights=None if weights is None else weights[mask],
        )[0]
    return out


def summarize(q2_edges, pn_edges, label='', threshold=MIN_MC_EVENTS):
    h = occupancy(q2_edges, pn_edges)
    mc = h['mc']
    return {
        'binning': label,
        'nQ2': len(q2_edges) - 1, 'npn': len(pn_edges) - 1, 'nbins': mc.size,
        'MC min': int(mc.min()), 'MC 5%': float(np.percentile(mc, 5)), 'MC median': float(np.median(mc)),
        f'MC<{threshold}': int((mc < threshold).sum()), 'MC<20': int((mc < 20).sum()),
        'signal zero': int((h['signal'] == 0).sum()), 'background zero': int((h['background'] == 0).sum()),
        'expected min': float(h['expected'].min()), 'expected<1': int((h['expected'] < 1).sum()),
        'NuWro min': int(h['nuwro'].min()),
    }


def worst_bins(q2_edges, pn_edges, n=8):
    '''The n least populated reco bins, with their edges and the other samples' counts.'''
    h = occupancy(q2_edges, pn_edges)
    order = np.argsort(h['mc'], axis=None)[:n]
    rows = []
    for flat in order:
        i, j = np.unravel_index(flat, h['mc'].shape)
        rows.append({
            'log10Q2': f'[{q2_edges[i]:.3g}, {q2_edges[i + 1]:.3g}]', 'p_n': f'[{pn_edges[j]:.3g}, {pn_edges[j + 1]:.3g}]',
            'MC raw': int(h['mc'][i, j]), 'signal': int(h['signal'][i, j]), 'background': int(h['background'][i, j]),
            'expected': round(float(h['expected'][i, j]), 1), 'NuWro raw': int(h['nuwro'][i, j]),
        })
    return pd.DataFrame(rows)


def plot_occupancy(q2_edges, pn_edges, title='', threshold=MIN_MC_EVENTS, samples=('mc', 'expected')):
    '''Annotated occupancy maps; bins below the raw-MC threshold are outlined in red.'''
    h = occupancy(q2_edges, pn_edges)
    fig, axes = plt.subplots(1, len(samples), figsize=(7.5 * len(samples), 5.5), squeeze=False)
    for ax, name in zip(axes[0], samples):
        values = h[name]
        norm = mpl.colors.LogNorm(vmin=max(values[values > 0].min(), 0.5), vmax=values.max())
        mesh = ax.pcolormesh(pn_edges, q2_edges, np.where(values > 0, values, np.nan), norm=norm, cmap='viridis')
        fig.colorbar(mesh, ax=ax, label=name)
        small = values.size <= 400
        for i in range(len(q2_edges) - 1):
            for j in range(len(pn_edges) - 1):
                if small:
                    text = f'{values[i, j]:.0f}'
                    ax.text(0.5 * (pn_edges[j] + pn_edges[j + 1]), 0.5 * (q2_edges[i] + q2_edges[i + 1]), text,
                            ha='center', va='center', fontsize=6, color='white' if values[i, j] < 0.3 * values.max() else 'black')
                if h['mc'][i, j] < threshold:
                    ax.add_patch(mpl.patches.Rectangle((pn_edges[j], q2_edges[i]), pn_edges[j + 1] - pn_edges[j],
                                                       q2_edges[i + 1] - q2_edges[i], fill=False, edgecolor='red', lw=1.8))
        ax.set_xlabel('$p_n$ [GeV]')
        ax.set_ylabel('$\\log_{10}(Q^2 / \\mathrm{GeV}^2)$')
        ax.set_title(f'{title} — {name}'.strip(' —'))
    fig.suptitle(f'{len(q2_edges) - 1} x {len(pn_edges) - 1} = {(len(q2_edges) - 1) * (len(pn_edges) - 1)} bins; '
                 f'red = fewer than {threshold} raw MC events', y=1.02)
    fig.tight_layout()
    plt.show()
    return h


def print_edges(q2_edges, pn_edges):
    '''The edges in the two formats used by the generator and the XMLs.'''
    print('python  q2 =', [float(f'{e:.4g}') for e in q2_edges])
    print('python  pn =', [float(f'{e:.4g}') for e in pn_edges])
    print(f'xml     edgesx="{fmt_edges(q2_edges)}"')
    print(f'xml     edgesy="{fmt_edges(pn_edges)}"')

## Section 1: the study variants

The summary table is the notebook equivalent of `make_binning_study_xmls.py --check`, computed
here from the edges in the generated XMLs. `nominal` is the production binning and is exempt
from the threshold: its corner bin (log10(Q^2) in [-2, -1.5], p_n in [0, 0.1]) holds 5 MC
events, which is why the finer variants start their p_n axis at [0, 0.15].

In [ ]:
SUMMARY = pd.DataFrame([summarize(q2, pn, name) for name, (q2, pn) in VARIANTS.items()]).set_index('binning')
display(SUMMARY)

for name, (q2, pn) in VARIANTS.items():
    below = int((occupancy(q2, pn)['mc'] < MIN_MC_EVENTS).sum())
    status = 'ok' if below == 0 else ('exempt' if name in THRESHOLD_EXEMPT else 'FAILS')
    print(f'{name:14s} {status:7s} log10Q2: {fmt_edges(q2)}\n{"":22s} p_n:     {fmt_edges(pn)}')

In [ ]:
for name, (q2, pn) in VARIANTS.items():
    plot_occupancy(q2, pn, title=name)
    display(worst_bins(q2, pn, n=5))

## Section 2: where is there room to split?

Marginal densities on a fine uniform grid, with the nominal edges drawn on top. Bins can only be
split where the marginal is high **and** the sparse corners (lowest Q^2 row, highest p_n column)
still keep enough events, so also look at the per-row and per-column minima below.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].hist(LOGQ2[overlay], bins=np.arange(-2.0, 0.2001, 0.05), histtype='stepfilled', alpha=0.6, label='MC raw')
axes[0].hist(LOGQ2[nuwro], bins=np.arange(-2.0, 0.2001, 0.05), histtype='step', color='k', label='NuWro raw')
for e in NOMINAL_Q2:
    axes[0].axvline(e, color='red', lw=0.8, alpha=0.7)
axes[0].set_xlabel('$\\log_{10}(Q^2 / \\mathrm{GeV}^2)$'); axes[0].set_ylabel('events / 0.05'); axes[0].legend()
axes[1].hist(PN[overlay], bins=np.arange(0, 1.0001, 0.025), histtype='stepfilled', alpha=0.6, label='MC raw')
axes[1].hist(PN[nuwro], bins=np.arange(0, 1.0001, 0.025), histtype='step', color='k', label='NuWro raw')
for e in NOMINAL_PN:
    axes[1].axvline(e, color='red', lw=0.8, alpha=0.7)
axes[1].set_xlabel('$p_n$ [GeV]'); axes[1].set_ylabel('events / 0.025'); axes[1].legend()
fig.suptitle('Marginal distributions of the selected events; red lines = production edges')
fig.tight_layout(); plt.show()

# Sparsest cell in each nominal row / column: a row (column) can only be split if its
# sparsest cell keeps >= 2 x MIN_MC_EVENTS events.
h_nom = occupancy(NOMINAL_Q2, NOMINAL_PN)['mc']
display(pd.DataFrame({
    'log10Q2 bin': [f'[{lo:.2f}, {hi:.2f}]' for lo, hi in zip(NOMINAL_Q2[:-1], NOMINAL_Q2[1:])],
    'row total': h_nom.sum(axis=1).astype(int), 'row min cell': h_nom.min(axis=1).astype(int),
}).set_index('log10Q2 bin').T)
display(pd.DataFrame({
    'p_n bin': [f'[{lo:.2f}, {hi:.2f}]' for lo, hi in zip(NOMINAL_PN[:-1], NOMINAL_PN[1:])],
    'column total': h_nom.sum(axis=0).astype(int), 'column min cell': h_nom.min(axis=0).astype(int),
}).set_index('p_n bin').T)

## Section 3: tune a candidate binning by hand

Edit `CANDIDATE_Q2` / `CANDIDATE_PN` and rerun. `halve(edges, keep=(...))` splits every bin
except the listed indices (negative indices count from the end). `prune_edges` starts from an
over-fine grid and repeatedly fixes the sparsest bin by removing one of its bounding interior
edges (on either axis, unless `axis` restricts it), preferring the removal that leaves the fewest
bins below `threshold`. It is a quick way to find the finest admissible binning; the result is a
starting point with irregular edges, not a substitute for choosing edges that make physical sense.

When the candidate is satisfactory, paste the printed `python` lines into `VARIANTS` in
`make_binning_study_xmls.py`, rerun it with `--check`, and run the new variant with
`xml/binning_study/run_all.sh --variant <name>`.

In [ ]:
def prune_edges(q2_edges, pn_edges, threshold=MIN_MC_EVENTS, axis='both', protect_q2=(), protect_pn=()):
    '''Remove interior edges until every reco bin holds >= threshold raw MC events.

    Each step looks at the sparsest bin and tries removing each of its bounding interior
    edges (log10 Q^2 and/or p_n, depending on `axis`). The removal that leaves the fewest
    bins below threshold wins; ties go to the removal that loses fewer bins, then to the
    larger resulting minimum. Edges listed in `protect_*` are never removed.
    '''
    q2_edges, pn_edges = list(q2_edges), list(pn_edges)
    while True:
        h = occupancy(q2_edges, pn_edges)['mc']
        if h.min() >= threshold:
            return q2_edges, pn_edges
        i, j = np.unravel_index(h.argmin(), h.shape)
        trials = []
        if axis in ('both', 'q2'):
            for e in (i, i + 1):
                if 0 < e < len(q2_edges) - 1 and q2_edges[e] not in protect_q2:
                    trial = occupancy(q2_edges[:e] + q2_edges[e + 1:], pn_edges)['mc']
                    trials.append((-(trial < threshold).sum(), -(len(pn_edges) - 1), trial.min(), 'q2', e))
        if axis in ('both', 'pn'):
            for e in (j, j + 1):
                if 0 < e < len(pn_edges) - 1 and pn_edges[e] not in protect_pn:
                    trial = occupancy(q2_edges, pn_edges[:e] + pn_edges[e + 1:])['mc']
                    trials.append((-(trial < threshold).sum(), -(len(q2_edges) - 1), trial.min(), 'pn', e))
        if not trials:
            raise RuntimeError(f'bin {i},{j} ({h[i, j]:.0f} events) has no removable bounding edge; '
                               'protect fewer edges or allow the other axis')
        *_, which, e = max(trials)
        if which == 'q2':
            del q2_edges[e]
        else:
            del pn_edges[e]


# --- edit here -------------------------------------------------------------
CANDIDATE_Q2 = halve(NOMINAL_Q2, keep=(0, -1))
CANDIDATE_PN = halve(STUDY_PN, keep=(0, 1, -1))
# Automatic route: start from quarter-width bins everywhere and prune both axes.
# CANDIDATE_Q2, CANDIDATE_PN = prune_edges(halve(halve(NOMINAL_Q2)), halve(halve(STUDY_PN)))
# ---------------------------------------------------------------------------

display(pd.DataFrame([summarize(CANDIDATE_Q2, CANDIDATE_PN, 'candidate')]).set_index('binning'))
plot_occupancy(CANDIDATE_Q2, CANDIDATE_PN, title='candidate')
display(worst_bins(CANDIDATE_Q2, CANDIDATE_PN))
print_edges(CANDIDATE_Q2, CANDIDATE_PN)

## Section 4: how far can the binning go?

Prune an over-fine start for several thresholds. Starting from the halved grid shows how close
the study variants are to the finest regular refinement; starting from quarter-width bins shows
how many bins the MC sample could support at all if the edges were free.

In [ ]:
rows = []
for start_name, (start_q2, start_pn) in {
    'halved start': (halve(NOMINAL_Q2), halve(STUDY_PN)),
    'quarter start': (halve(halve(NOMINAL_Q2)), halve(halve(STUDY_PN))),
}.items():
    for threshold in (5, 10, 20, 40):
        q2, pn = prune_edges(start_q2, start_pn, threshold=threshold)
        rows.append(summarize(q2, pn, f'{start_name}, threshold {threshold}') | {'threshold': threshold})
        if threshold == MIN_MC_EVENTS:
            print(f'{start_name}, pruned to threshold {threshold}:')
            print_edges(q2, pn)
            plot_occupancy(q2, pn, title=f'{start_name}, threshold {threshold}', samples=('mc',))
display(pd.DataFrame(rows).set_index('binning'))